# Le contexte rétrécit la borne — entropie conditionnelle et taux de compression

**Série Compression — carnet 09b — jumeau C# du [carnet Python](Compression-09b-Entropie-Conditionnelle-Taux-Compression-Python.ipynb)**

**Navigation** : [Index de la série](README.md) | [Compression-08 — Assemblage vectoriel d'un document](Compression-08-Assemblage-Vectoriel-Python.ipynb) | [jumeau Python](Compression-09b-Entropie-Conditionnelle-Taux-Compression-Python.ipynb)

Le carnet 01 a établi la borne de Shannon : un code préfixe ne descend jamais sous l'entropie H(X) des symboles, symbole par symbole. Le carnet 04 a pourtant exhibé des codes (ANS) qui descendent sous l'entropie des octets d'un texte. Pas de contradiction : l'entropie d'octet calcule H(X) sur une source **sans mémoire** — chaque octet vu isolément. Or une vraie source a de la mémoire : la lettre qui suit un « q » n'est pas distribuée comme la lettre qui suit un espace. Le README de la série le formule : *« un codec peut descendre sous l'entropie des octets calculée sans tenir compte de leur contexte »*.

Ce carnet donne le corps de cette phrase : **l'entropie conditionnelle** H(X|Y), **l'information mutuelle** I(X;Y), et surtout le **taux d'entropie** d'un processus — la borne de compression *réelle* d'une source à mémoire. La démarche est la même que dans 01 : on construit la mesure à la main, on la vérifie sur une source dont la valeur **analytique** est connue, puis on la lit sur des sources où elle n'est pas connue — avec un témoin négatif qui garantit que la méthode ne fabrique pas de structure là où il n'y en a pas.

**Prérequis** : le carnet 01 (entropie, code préfixe) et des probabilités discrètes. Le jumeau Python calcule avec numpy ; ce jumeau C# reconstruit les mêmes tables de probabilité et de contextes **à la main, avec la seule bibliothèque standard .NET** (`System.Math`, collections génériques) — aucun paquet NuGet. L'alphabet reste de 27 symboles (a–z et l'espace), petit à dessein : les tables de contextes doivent être bien remplies pour que les estimations restent honnêtes.

## 1. Rappel — l'entropie comme borne

Pour une source **sans mémoire** — chaque symbole tiré indépendamment selon une loi `p` — l'entropie

    H(X) = − Σ_x p(x) log2 p(x)

se mesure en bits par symbole. Le théorème de codage des sources (carnet 01) dit qu'aucun code préfixe ne descend, en moyenne, sous `H(X)` : c'est la borne que Huffman sature.

Cette borne est **trop pessimiste** dès que la source a de la mémoire. Le symbole présent y dépend du passé, et un code qui *exploite* ce passé descend sous `H(X)`. L'écart entre `H(X)` et ce que le codage à contexte atteint réellement est précisément ce que ce carnet va mesurer.

In [1]:
#nullable enable
using System;
using System.Collections.Generic;
using System.Globalization;
using System.Linq;

// Alphabet de taille 27 symboles : a-z (26 lettres) + espace (code 26)
public static class Info
{
    public const int Alphabet = 27;

    // Entropie de Shannon (bits) d'une loi donnee par ses effectifs (seuls les effectifs > 0 comptent)
    public static double EntropieDe(IEnumerable<int> counts)
    {
        double total = 0.0;
        foreach (int c in counts) total += c;
        double h = 0.0;
        foreach (int c in counts)
        {
            if (c > 0)
            {
                double p = c / total;
                h -= p * Math.Log2(p);
            }
        }
        return h;
    }

    // Entropie par symbole H(X) - contexte de longueur 0 (source sans memoire)
    public static double H0(int[] s)
    {
        var counts = new int[Alphabet];
        foreach (int x in s) counts[x]++;
        return EntropieDe(counts);
    }

    // Entropie conditionnelle H(X_t | X_{t-1} ... X_{t-order}) en bits/symbole.
    // On compte les paires (contexte de `order` symboles, symbole suivant) puis on
    // moyenne, pondere par P(contexte), l'entropie du symbole suivant. On stocke les
    // effectifs dans des dictionnaires : l'alphabet de 27 symboles rend le tableau
    // dense 27^order * 27 trop gros des l'ordre 4-5, alors que les contextes
    // effectivement observes restent peu nombreux dans l'echantillon.
    public static double HConditionnelle(int[] s, int order)
    {
        if (s.Length <= order) return double.NaN;
        var jointes = new Dictionary<long, long>();
        var contexto = new Dictionary<long, long>();
        long total = 0;
        for (int t = order; t < s.Length; t++)
        {
            long ctx = 0;
            for (int k = 0; k < order; k++) ctx = ctx * Alphabet + s[t - order + k];
            long cle = ctx * Alphabet + s[t];
            long v;
            jointes[cle] = jointes.TryGetValue(cle, out v) ? v + 1 : 1;
            long cv;
            contexto[ctx] = contexto.TryGetValue(ctx, out cv) ? cv + 1 : 1;
            total++;
        }
        double h = 0.0;
        foreach (var kv in jointes)
        {
            long cell = kv.Value;
            long ctx = kv.Key / Alphabet;
            long cc = contexto[ctx];
            double pJointe = (double)cell / total;
            double pCond = (double)cell / cc;
            h -= pJointe * Math.Log2(pCond);
        }
        return h;
    }

    // I(X_t ; X_{t-1}) = H(X_t) - H(X_t | X_{t-1}), en bits/symbole
    public static double InfoMutuelleOrdre1(int[] s)
    {
        var jointe = new long[Alphabet * Alphabet];
        for (int t = 1; t < s.Length; t++) jointe[s[t - 1] * Alphabet + s[t]]++;
        double total = 0.0;
        for (int i = 0; i < jointe.Length; i++) total += jointe[i];
        var pp = new double[Alphabet];
        var pc = new double[Alphabet];
        for (int i = 0; i < Alphabet; i++)
        {
            long ligne = 0, colonne = 0;
            for (int j = 0; j < Alphabet; j++)
            {
                ligne += jointe[i * Alphabet + j];
                colonne += jointe[j * Alphabet + i];
            }
            pp[i] = ligne / total;
            pc[i] = colonne / total;
        }
        double mi = 0.0;
        for (int i = 0; i < Alphabet; i++)
            for (int j = 0; j < Alphabet; j++)
            {
                long c = jointe[i * Alphabet + j];
                if (c == 0) continue;
                double pj = c / total;
                double denom = pp[i] * pc[j];
                mi += pj * Math.Log2(pj / denom);
            }
        return mi;
    }

    // Tirage pondere (equivalent de np.random.choice(A, p=w))
    public static int TiragePondere(Random rng, double[] poids)
    {
        double u = rng.NextDouble();
        double cumul = 0.0;
        for (int i = 0; i < poids.Length; i++)
        {
            cumul += poids[i];
            if (u < cumul) return i;
        }
        return poids.Length - 1;
    }
}

// La culture invariante garantit le separateur decimal "." dans toutes les sorties
CultureInfo.CurrentCulture = CultureInfo.InvariantCulture;
CultureInfo.DefaultThreadCurrentCulture = CultureInfo.InvariantCulture;

Console.WriteLine($"Fonctions d'information definies (alphabet de {Info.Alphabet} symboles).");

Fonctions d'information definies (alphabet de 27 symboles).


## 2. L'entropie conditionnelle H(X | Y)

L'entropie conditionnelle d'une variable `X` sachant `Y` est

    H(X | Y) = Σ_y p(y) · H(X | Y = y)

C'est l'incertitude **résiduelle** sur `X` une fois `Y` connue. Si `Y` est informatif, `H(X | Y) < H(X)` ; si `Y` est sans lien avec `X`, alors `H(X | Y) = H(X)`.

L'écart est l'**information mutuelle** :

    I(X ; Y) = H(X) − H(X | Y) = H(Y) − H(Y | X)  ≥  0

Elle mesure ce que savoir `Y` enlève d'incertitude sur `X`, et est nulle exactement quand `X` et `Y` sont indépendants. Appliquée à une source, `Y` est le **passé** et `X` le **présent** : l'information mutuelle quantifie ce que le contexte porte déjà sur le symbole à venir.

## 3. Le taux d'entropie d'une source à mémoire

Pour une source qui a de la mémoire, on ne se contente plus de `H(X)`. On regarde combien d'information le passé de longueur `k` enlève au présent :

    H(X_t | X_{t-1} ... X_{t-k})

Quand `k` croît, cette quantité **décroît** — on sait de plus en plus — et converge vers le **taux d'entropie** :

    ρ = lim_{k→∞} H(X_t | X_{t-1} ... X_{t-k})

`ρ` est la borne de compression **réelle** d'une source stationnaire : le nombre de bits/symbole qu'aucun code ne descend en moyenne, une fois tout le contexte exploité. Pour une source sans mémoire, `ρ = H(X)`. Pour une source à mémoire, `ρ < H(X)` — et c'est exactement le terrain que LZ77 (longues fenêtres de contexte, carnet 03) et ANS (modèle de distribution, carnet 04) exploitent pour descendre sous l'entropie par symbole.

## 4. Témoin rigoureux — une chaîne de Markov au taux d'entropie connu

Pour que la mesure soit **falsifiable**, on la confronte d'abord à une source dont le taux d'entropie se calcule à la main. Prenons une chaîne de Markov à deux états :

    P = [[0.9, 0.1],     # depuis l'état 0 : 90 % rester, 10 % changer
         [0.5, 0.5]]     # depuis l'état 1 : équiprobable

Son taux d'entropie **analytique** est `ρ = Σ_i π_i · H(ligne_i)`, où `π` est la distribution stationnaire et `H(ligne_i)` l'entropie de la i-ème ligne de `P`. On le calcule ci-dessous, puis on vérifie que l'estimation empirique `H(· | 1)` le retrouve — le contrat de toute la suite du carnet.

In [2]:
// --- Chaine de Markov a 2 etats : taux d'entropie analytique vs estimation ---
double[,] P = { { 0.9, 0.1 }, { 0.5, 0.5 } };

// Distribution stationnaire : on itere pi = pi @ P jusqu'a convergence
double[] pi = { 1.0, 0.0 };
for (int it = 0; it < 200; it++)
{
    double[] suivant = { pi[0] * P[0, 0] + pi[1] * P[1, 0], pi[0] * P[0, 1] + pi[1] * P[1, 1] };
    pi = suivant;
}
double sommePi = pi[0] + pi[1];
pi[0] /= sommePi;
pi[1] /= sommePi;

// Entropie de chaque ligne de P, puis taux d'entropie analytique
double[] entropieLignes = new double[2];
for (int i = 0; i < 2; i++)
{
    double h = 0.0;
    for (int j = 0; j < 2; j++)
    {
        double p = P[i, j];
        if (p > 0) h -= p * Math.Log2(p);
    }
    entropieLignes[i] = h;
}
double rhoAnalytique = pi[0] * entropieLignes[0] + pi[1] * entropieLignes[1];
double h0Analytique = 0.0;
for (int i = 0; i < 2; i++) if (pi[i] > 0) h0Analytique -= pi[i] * Math.Log2(pi[i]);

// --- Generation d'un long echantillon (dependance d'etat sequentielle) ---
var rngMarkov = new Random(42);
int nMarkov = 200_000;
int[] echMarkov = new int[nMarkov];
for (int t = 1; t < nMarkov; t++)
    echMarkov[t] = (rngMarkov.NextDouble() < P[echMarkov[t - 1], 0]) ? 0 : 1;

double h0M = Info.H0(echMarkov);
double h1M = Info.HConditionnelle(echMarkov, 1);
double h2M = Info.HConditionnelle(echMarkov, 2);

var invM = CultureInfo.InvariantCulture;
Console.WriteLine("Chaîne de Markov (2 états) :");
Console.WriteLine($"  distribution stationnaire  pi = [{pi[0].ToString("F4", invM)} {pi[1].ToString("F4", invM)}]");
Console.WriteLine($"  H0   analytique = {h0Analytique.ToString("F4", invM)}    |  empirique = {h0M.ToString("F4", invM)}");
Console.WriteLine($"  rho  analytique = {rhoAnalytique.ToString("F4", invM)}    |  empirique H(·|1) = {h1M.ToString("F4", invM)}");
Console.WriteLine($"  H(·|2) empirique = {h2M.ToString("F4", invM)}");
string signeEcart = (h1M - rhoAnalytique) >= 0 ? "+" : "";
Console.WriteLine($"  écart H(·|1) - rho = {signeEcart}{(h1M - rhoAnalytique).ToString("F4", invM)}  (la mesure est-elle juste ?)");

Chaîne de Markov (2 états) :


  distribution stationnaire  pi = [0.8333 0.1667]


  H0   analytique = 0.6500    |  empirique = 0.6504


  rho  analytique = 0.5575    |  empirique H(·|1) = 0.5567


  H(·|2) empirique = 0.5567


  écart H(·|1) - rho = -0.0008  (la mesure est-elle juste ?)


### Lecture du témoin

L'estimation `H(· | 1)` **colle au taux d'entropie analytique** (écart de l'ordre de 10⁻³) : la mesure est juste. Et `H(· | 2)` **n'abaisse plus la borne** — une chaîne de Markov d'ordre 1 n'a de mémoire que d'un pas, donc son taux est atteint dès qu'on conditionne sur le symbole précédent. Le contexte supplémentaire n'apporte rien : c'est le signal qu'on a atteint `ρ`, pas un artefact d'échantillonnage. Ce point d'appui — une valeur connue à la main, retrouvée par la mesure — est ce qui autorise à lire des sources dont la borne n'est *pas* connue.

## 5. La courbe du taux d'entropie

La quantité `H(X_t | X_{t-1} ... X_{t-k})` en fonction de la longueur de contexte `k` est la **courbe du taux d'entropie**. Elle est décroissante — plus de contexte ne peut qu'informer, jamais dégrader — et devient **plate** dès que tout le contexte utile est inclus. Son plateau **est** `ρ`, la borne de compression fondamentale de la source.

Le tableau ci-dessous donne cette courbe pour la chaîne de Markov (plateau net à l'ordre 1, la borne analytique rappelée à côté), pour une source **structurée** — où le contexte abaisse bien la borne — et pour une source **i.i.d.** — où, faute de structure, aucun abaissement **réel** n'apparaît. Le léger repli mesuré à l'ordre 2 n'est pas de la structure : c'est le biais à la baisse de l'estimateur d'entropie par maximum de vraisemblance. À l'ordre 2, les 27² contextes possibles sont peu peuplés dans un échantillon de 120 000 symboles, et la moyenne ne porte que sur les paires (contexte, symbole) effectivement observées — ce qui sous-estime systématiquement l'entropie vraie. Ce repli croît avec l'ordre du contexte et la taille de l'alphabet rapportée à l'échantillon ; c'est exactement le piège d'échantillonnage que l'exercice 3 isole. Le jumeau C# rend ce graphe sous forme de tableau texte (le noyau `.net-csharp` ne fournit pas matplotlib sans dépendance).

In [3]:
// --- Sources de controle : structuree (Zipf + memoire de repetition) et i.i.d. uniforme ---
double[] poidsZipf = new double[Info.Alphabet];
double sommeZipf = 0.0;
for (int i = 0; i < Info.Alphabet; i++) { poidsZipf[i] = 1.0 / (i + 1); sommeZipf += poidsZipf[i]; }
for (int i = 0; i < Info.Alphabet; i++) poidsZipf[i] /= sommeZipf;

var rngTxt = new Random(7);
int mTxt = 120_000;
int[] sourceTxt = new int[mTxt];
sourceTxt[0] = Info.TiragePondere(rngTxt, poidsZipf);
for (int t = 1; t < mTxt; t++)
    sourceTxt[t] = (rngTxt.NextDouble() < 0.5) ? sourceTxt[t - 1] : Info.TiragePondere(rngTxt, poidsZipf);

var rngIid = new Random(11);
int[] sourceIid = new int[mTxt];
for (int t = 0; t < mTxt; t++) sourceIid[t] = rngIid.Next(0, Info.Alphabet);

int[] ordres = { 0, 1, 2, 3, 4, 5 };
double[] Courbe(int[] src)
{
    var vals = new double[ordres.Length];
    for (int i = 0; i < ordres.Length; i++)
        vals[i] = ordres[i] == 0 ? Info.H0(src) : Info.HConditionnelle(src, ordres[i]);
    return vals;
}
double[] courbeMarkov = Courbe(echMarkov);
double[] courbeTxt = Courbe(sourceTxt);
double[] courbeIid = Courbe(sourceIid);

var invC = CultureInfo.InvariantCulture;
Console.WriteLine("Taux d'entropie H(X_t | contexte de longueur k) — trois sources (bits/symbole) :");
Console.WriteLine("  k | Markov(2 etats) |    structuree |        i.i.d.");
Console.WriteLine(" ---+----------------+---------------+--------------");
for (int i = 0; i < ordres.Length; i++)
{
    string cm = courbeMarkov[i].ToString("F3", invC).PadLeft(14);
    string ct = courbeTxt[i].ToString("F3", invC).PadLeft(13);
    string ci = courbeIid[i].ToString("F3", invC).PadLeft(13);
    Console.WriteLine($"  {ordres[i]} | {cm} | {ct} | {ci}");
}

Taux d'entropie H(X_t | contexte de longueur k) — trois sources (bits/symbole) :


  k | Markov(2 etats) |    structuree |        i.i.d.


 ---+----------------+---------------+--------------


  0 |          0.650 |         3.969 |         4.755


  1 |          0.557 |         2.739 |         4.751


  2 |          0.557 |         2.651 |         4.637


  3 |          0.557 |         2.321 |         2.515


  4 |          0.557 |         1.820 |         0.207


  5 |          0.557 |         1.303 |         0.008


In [4]:
// Bilan : une ligne de synthese par source — H0 (entropie marginale),
// H(.|1) et H(.|2) (entropie conditionnelle, la borne par symbole),
// I(.;.) (information mutuelle d'ordre 1) et l'abaissement 0 -> 1.
void Bilan(int[] src, string nom)
{
    double h0v = Info.H0(src);                // entropie marginale H(X)
    double h1v = Info.HConditionnelle(src, 1); // H(X_t | X_{t-1})
    double h2v = Info.HConditionnelle(src, 2); // H(X_t | X_{t-2..t-1})
    double im = Info.InfoMutuelleOrdre1(src);  // I(X_t ; X_{t-1}) = H0 - H(.|1)
    var invB = CultureInfo.InvariantCulture;
    string s0 = h0v.ToString("F3", invB).PadLeft(6);
    string s1 = h1v.ToString("F3", invB).PadLeft(6);
    string s2 = h2v.ToString("F3", invB).PadLeft(6);
    string sim = im.ToString("F3", invB).PadLeft(5);
    string sAb = (h0v - h1v).ToString("F3", invB).PadLeft(6);
    Console.WriteLine($"{nom.PadLeft(18)} :  H0={s0}   H(·|1)={s1}   H(·|2)={s2}   I(·;·)={sim}   abaissement 0->1={sAb}");
}

// Le contraste structuree vs i.i.d. : la structure abaisse la borne, l'i.i.d. non.
Console.WriteLine("Borne par symbole (bits) selon la présence ou non de structure :");
Console.WriteLine();
Bilan(sourceTxt, "source structurée");
Bilan(sourceIid, "source i.i.d.");
Console.WriteLine();
// Plafond uniforme : sans memoire, H ne descend pas sous log2(|A|).
Console.WriteLine($"Rappel : log2(27) = {Math.Log2(Info.Alphabet).ToString("F3", CultureInfo.InvariantCulture)} bits — le plafond d'une source uniforme sans mémoire.");

Borne par symbole (bits) selon la présence ou non de structure :


 source structurée :  H0= 3.969   H(·|1)= 2.739   H(·|2)= 2.651   I(·;·)=1.230   abaissement 0->1= 1.230


     source i.i.d. :  H0= 4.755   H(·|1)= 4.751   H(·|2)= 4.637   I(·;·)=0.004   abaissement 0->1= 0.004


Rappel : log2(27) = 4.755 bits — le plafond d'une source uniforme sans mémoire.


### Lecture du contraste

Sur la source **structurée**, la connaissance du symbole précédent fait tomber la borne de ~4,0 à ~2,8 bits/symbole : le voisin porte près d'un tiers de l'incertitude. Sur la source **i.i.d.**, rien ne bouge **de l'ordre 0 à l'ordre 1** — H0 ≈ H(·|1) ≈ log2(27) : sans structure, aucun contexte ne descend la borne, et la mesure ne **fabrique** pas de structure fictive. Le repli à l'ordre 2 (0,116 bit) est le biais d'estimation vu au §5, pas de la structure. Ce témoin négatif est ce qui rend l'abaissement observé sur la source structurée **attribuable à la mémoire** et non à un biais d'estimation. C'est le même geste qu'au carnet 04, où les octets aléatoires servaient de contrôle au banc de codecs.

## 6. Où cela se retrouve dans la série

Cette géométrie — *borne par symbole* en haut, *borne réelle* en bas, le codec entre les deux — éclaire en retour les carnets précédents :

- **03 (LZ77)** remplace une répétition par une référence dans une fenêtre : la fenêtre est un **contexte**, et la copie est exactement la prédiction « ce symbole était prévisible sachant le contexte ». Le carnet 03 observait qu'un codec descend sous l'entropie des octets : le présent carnet donne la mesure de la borne qu'il approche.
- **04 (ANS)** code chaque symbole en lui donnant exactement `−log2 p(s)` bits, où `p` est la probabilité **modélisée**. Quand le modèle est conditionnel (porté par l'état du codeur), c'est `H(X | contexte)` que ANS sature, pas `H(X)` — c'est le mécanisme précis par lequel la sortie de ~0,09 bit/symbole du carnet 04 est possible.
- **01/01b (Huffman, Kraft)** restent exacts dans leur cadre : pour une source **sans mémoire**, la borne est `H(X)` et Huffman la sature. Ce carnet ne les contredit pas, il explicite leur hypothèse.

Le lecteur qui veut pousser vers la théorie de l'estimation trouvera dans le carnet 06 le prolongement algorithmique : la complexité de Kolmogorov est la borne ultime, **non calculable** — le taux d'entropie en est la version calculable, pour les sources stationnaires.

## 7. Exercices

Les trois exercices prolongent la démarche du carnet : mesurer, contrôler, confronter à une valeur connue. Chaque cellule attend une implémentation (`return null` par défaut) ; le carnet s'exécute de bout en bout même non complétés.

In [5]:
// Exercice 1 — entropie conditionnelle par empilement
// On a mesure H(X_t | X_{t-1}) avec la table jointe complete. Ecrire une fonction
// qui calcule la meme chose en iterant sur les contextes observes : pour chaque
// contexte c rencontre au moins deux fois, entropie locale du symbole suivant,
// ponderee par la frequence du contexte. Verifier l'egalite des deux methodes
// sur la source structuree (ecart < 1e-9).
double? HConditionnelleParContexte(int[] s)
{
    // TODO etudiant : construire les contextes, grouper par contexte,
    // moyenne ponderee des entropies locales
    return null;
}

double? resultatExercice1 = HConditionnelleParContexte(sourceTxt);
Console.WriteLine("Exercice 1 a completer : HConditionnelleParContexte(s)");

Exercice 1 a completer : HConditionnelleParContexte(s)


In [6]:
// Exercice 2 — le plateau de la courbe
// La courbe H(X_t | contexte de longueur k) est decroissante puis plate.
// Ecrire une fonction qui, pour une source donnee, renvoie l'ordre k* au-dela
// duquel le gain marginal H(·|k-1) - H(·|k) passe sous un seuil (par defaut
// 0.01 bit). Tester sur : la chaine de Markov (attendu k* = 1), la source
// structuree (quel est son ordre memoire reel ?), la source i.i.d. (attendu k* = 0).
int? OrdreMemoire(int[] src, double seuil = 0.01, int kMax = 6)
{
    // TODO etudiant : balayer k, comparer le gain marginal au seuil
    return null;
}

int? resultatExercice2 = OrdreMemoire(sourceTxt);
Console.WriteLine("Exercice 2 a completer : OrdreMemoire(src)");

Exercice 2 a completer : OrdreMemoire(src)


In [7]:
// Exercice 3 — le controle negatif et le biais d'echantillonnage
// La source i.i.d. uniforme ne doit montrer AUCUNE reduction en fonction de
// l'ordre k : c'est ce qui prouve que la methode n'invente pas de structure.
// a) Verifier sur le tableau du §5 que la courbe i.i.d. est plate autour de log2(27).
// b) Compter, pour la source i.i.d., combien de contextes d'ordre 1 (27 possibles)
//    et d'ordre 3 (27^3 possibles) sont effectivement observes dans l'echantillon.
//    Expliquer pourquoi un alphabet trop grand (256 octets) avec un echantillon
//    fin fait s'effondrer faussement l'estimation vers 0.
int? ContextesObserves(int[] s, int order)
{
    // TODO etudiant : compter les contextes distincts reellement vus
    return null;
}

int? resultatExercice3 = ContextesObserves(sourceIid, 1);
Console.WriteLine("Exercice 3 a completer : ContextesObserves(s, order)");

Exercice 3 a completer : ContextesObserves(s, order)


## Conclusion

Le carnet a donné un corps au principe annoncé par la série : **le contexte rétrécit la borne**. L'entropie par symbole H(X) du carnet 01 est la borne d'une source **sans mémoire** ; une source à mémoire est bornée par son **taux d'entropie** ρ = limₖ H(X_t | X_{t-1}…X_{t-k}), toujours plus bas. Le témoin rigoureux l'établit sans approximation cachée : une chaîne de Markov dont le taux analytique est connu à l'avance, que l'estimation empirique retrouve à ~10⁻³ près, avec une courbe qui s'aplatit exactement à l'ordre mémoire réel. Le contrôle négatif (source i.i.d. plate) montre que la méthode n'invente pas de structure — et l'exercice 3 explique le piège d'échantillonnage qui la ferait mentir sur un alphabet trop grand.

C'est ce ρ, et non H(X), que LZ77 (03) et ANS (04) approchent en pratique : LZ77 avec ses contextes de longue portée, ANS en codant chaque symbole sur le nombre d'octets que la distribution conditionnelle lui alloue. Les carnets 01 et 01b restaient exacts — mais dans leur cadre sans mémoire, dont la borne était bien l'entropie par symbole.

**Navigation** : [Index de la série](README.md) | [Compression-08 — Assemblage vectoriel d'un document](Compression-08-Assemblage-Vectoriel-Python.ipynb) | [jumeau Python](Compression-09b-Entropie-Conditionnelle-Taux-Compression-Python.ipynb)